In [55]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# __Section A - Coding__

In this section, we implement Decision Tree and Random Forest models from scratch for both classification and regression.

**Node Class** - this class represents a single node in the tree.   It stores the node impurity, the prediction, the split rule, and links to the child nodes.  
The same class is used for both classification and regression trees.

In [56]:
# Class for one node in a tree
class Node:
    def __init__(
        self,
        impurity=None,
        num_samples=None,
        prediction=None,
        num_samples_per_class=None,
        feature_index=None,
        threshold=None,
        left=None,
        right=None
    ):
        # impurity at this node
        self.impurity = impurity

        # number of samples in this node
        self.num_samples = num_samples

        # stored prediction:
        self.prediction = prediction

        # class counts (used in classification if needed)
        self.num_samples_per_class = num_samples_per_class

        # split rule
        self.feature_index = feature_index
        self.threshold = threshold

        # children
        self.left = left
        self.right = right

    def is_leaf_node(self):
        return self.left is None and self.right is None

**Gini Impurity** - this function computes the impurity of a node in the classification tree.  
It measures how mixed the class labels are inside the node.  
Formula: **Gini = 1 - Σ(pᵢ²)**.

In [57]:
# Gini impurity calculation function
def calculate_gini(y):
    n = len(y)

    if n == 0:
        return 0.0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / n

    return 1.0 - np.sum(probabilities ** 2)

**SSR Calculation** - this function computes the impurity of a node in the regression tree.  
It measures how far the target values are from their mean inside the node.  
Formula: **SSR = Σ(yᵢ - ȳ)²**.

In [58]:
# SSR calculation
def calculate_ssr(y):
    n = len(y)

    if n == 0:
        return 0.0

    mean_value = np.mean(y)
    ssr = np.sum((y - mean_value) ** 2)

    return ssr

**Best Split** - this function searches for the best feature and threshold to split the node. For classification it uses Gini gain, and for regression it uses SSR reduction.  
It also enforces the minimum samples per leaf rule.

In [59]:
# Find the best split for classification or regression
def find_best_split(X, y, feature_names, min_samples_leaf=5, is_classifier=True):
    n_samples, n_features = X.shape

    if n_samples <= 1:
        return None, None, 0.0

    # parent impurity
    if is_classifier:
        parent_score = calculate_gini(y)
    else:
        parent_score = calculate_ssr(y)

    best_gain = 0.0
    best_feature_index = None
    best_threshold = None

    # sort features alphabetically
    sorted_feature_indices = sorted(range(n_features), key=lambda i: feature_names[i])

    for feature_index in sorted_feature_indices:
        values = X[:, feature_index]
        unique_values = np.unique(values)

        # no split possible
        if len(unique_values) < 2:
            continue

        # candidate thresholds
        thresholds = (unique_values[:-1] + unique_values[1:]) / 2

        for threshold in thresholds:
            left_mask = values <= threshold
            right_mask = values > threshold

            # skip small leaves
            if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
                continue

            left_y = y[left_mask]
            right_y = y[right_mask]

            if is_classifier:
                left_score = calculate_gini(left_y)
                right_score = calculate_gini(right_y)

                child_score = (
                    (len(left_y) / n_samples) * left_score
                    + (len(right_y) / n_samples) * right_score
                )
            else:
                left_score = calculate_ssr(left_y)
                right_score = calculate_ssr(right_y)

                child_score = left_score + right_score

            gain = parent_score - child_score

            # keep the first alphabetical feature
            if gain > best_gain:
                best_gain = gain
                best_feature_index = feature_index
                best_threshold = threshold

    return best_feature_index, best_threshold, best_gain

### __Decision Tree__

**Decision Tree Classifier** - this class implements a classification tree using Gini impurity.  
It predicts the majority class in each node.

In [60]:
class MyDecisionTreeClassifier:
    def __init__(self, min_samples_leaf=5, max_depth=None):
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.root = None
        self.feature_names = None

    # train the model: prepare the input data, save the feature names, and start building the tree
    def fit(self, X, y, feature_names=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names = X.columns.tolist() if feature_names is None else feature_names
            X = X.values
        else:
            self.feature_names = feature_names if feature_names is not None else [
                f"feat_{i}" for i in range(X.shape[1])
            ]

        # convert y when needed
        if isinstance(y, pd.Series):
            y = y.values
        else:
            y = np.array(y)

        self.root = self._build_tree(X, y, depth=0)
        return self

    # predict the class for each sample in X
    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        return np.array([self._traverse_tree(x, self.root) for x in X])

    # recursively build the tree
    def _build_tree(self, X, y, depth):
        num_samples = X.shape[0]

        # compute impurity and predicted class
        impurity = calculate_gini(y)
        values, counts = np.unique(y, return_counts=True)
        prediction = values[np.argmax(counts)] if len(y) > 0 else None
        num_samples_per_class = counts

        # stop conditions
        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or num_samples < 2 * self.min_samples_leaf
            or impurity == 0.0
        ):
            return Node(
                impurity=impurity,
                num_samples=num_samples,
                prediction=prediction,
                num_samples_per_class=num_samples_per_class
            )

        # find the best split
        best_feat, best_thresh, best_gain = find_best_split(
            X,
            y,
            self.feature_names,
            self.min_samples_leaf,
            is_classifier=True
        )

        # return a leaf if no valid split was found
        if best_feat is None or best_gain <= 0.0:
            return Node(
                impurity=impurity,
                num_samples=num_samples,
                prediction=prediction,
                num_samples_per_class=num_samples_per_class
            )

        # split the data
        left_mask = X[:, best_feat] <= best_thresh
        right_mask = X[:, best_feat] > best_thresh

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(
            impurity=impurity,
            num_samples=num_samples,
            prediction=prediction,
            num_samples_per_class=num_samples_per_class,
            feature_index=best_feat,
            threshold=best_thresh,
            left=left_child,
            right=right_child
        )

    # follow the tree until reaching a leaf
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.prediction

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

**Decision Tree Regressor** - this class implements a regression tree using SSR reduction.  
It predicts the mean target value in each node.

In [61]:
class MyDecisionTreeClassifier:
    def __init__(self, min_samples_leaf=5, max_depth=None):
        self.min_samples_leaf = min_samples_leaf
        self.max_depth = max_depth
        self.root = None
        self.feature_names = None

    # train the model: prepare the input data, save the feature names, and start building the tree
    def fit(self, X, y, feature_names=None):
        if isinstance(X, pd.DataFrame):
            self.feature_names = X.columns.tolist() if feature_names is None else feature_names
            X = X.values
        else:
            self.feature_names = feature_names if feature_names is not None else [
                f"feat_{i}" for i in range(X.shape[1])
            ]

        # convert y when needed
        if isinstance(y, pd.Series):
            y = y.values
        else:
            y = np.array(y)

        self.root = self._build_tree(X, y, depth=0)
        return self

    # predict the class for each sample in X
    def predict(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.values

        return np.array([self._traverse_tree(x, self.root) for x in X])

    # recursively build the tree
    def _build_tree(self, X, y, depth):
        num_samples = X.shape[0]

        # compute impurity and predicted class
        impurity = calculate_gini(y)
        values, counts = np.unique(y, return_counts=True)
        prediction = values[np.argmax(counts)] if len(y) > 0 else None
        num_samples_per_class = counts

        # stop conditions
        if (
            (self.max_depth is not None and depth >= self.max_depth)
            or num_samples < 2 * self.min_samples_leaf
            or impurity == 0.0
        ):
            return Node(
                impurity=impurity,
                num_samples=num_samples,
                prediction=prediction,
                num_samples_per_class=num_samples_per_class
            )

        # find the best split
        best_feat, best_thresh, best_gain = find_best_split(
            X,
            y,
            self.feature_names,
            self.min_samples_leaf,
            is_classifier=True
        )

        # return a leaf if no valid split was found
        if best_feat is None or best_gain <= 0.0:
            return Node(
                impurity=impurity,
                num_samples=num_samples,
                prediction=prediction,
                num_samples_per_class=num_samples_per_class
            )

        # split the data
        left_mask = X[:, best_feat] <= best_thresh
        right_mask = X[:, best_feat] > best_thresh

        left_child = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return Node(
            impurity=impurity,
            num_samples=num_samples,
            prediction=prediction,
            num_samples_per_class=num_samples_per_class,
            feature_index=best_feat,
            threshold=best_thresh,
            left=left_child,
            right=right_child
        )

    # follow the tree until reaching a leaf
    def _traverse_tree(self, x, node):
        if node.is_leaf_node():
            return node.prediction

        if x[node.feature_index] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [5]:
# Helper functions for Random Forest

import numpy as np


def bootstrap_sample(X, y):
    """
    Create a bootstrap sample (sampling with replacement).
    The sample has the same size as the original dataset.
    Works with pandas DataFrame and Series.
    """
    n_samples = X.shape[0]

    # sample row indices with replacement
    indices = np.random.choice(n_samples, size=n_samples, replace=True)

    X_resampled = X.iloc[indices]
    y_resampled = y.iloc[indices]

    return X_resampled, y_resampled

def majority_vote(predictions):
    """
    Return the most common class among predictions.
    Used in classification.
    """
    values, counts = np.unique(predictions, return_counts=True)
    return values[np.argmax(counts)]


def average_predictions(predictions):
    """
    Return the average of predictions.
    Used in regression.
    """
    return np.mean(predictions)

In [6]:
# Random Forest Classifier

class RandomForestClassifier:
    """
    Random Forest model for classification.
    The model trains multiple decision trees on bootstrap samples
    and combines their predictions using majority voting.
    """

    def __init__(self, n_estimators=10, max_depth=None, min_samples_leaf=5):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.trees = []

    def fit(self, X, y):
        """
        Train the random forest classifier.
        Each tree is trained on a different bootstrap sample.
        """
        self.trees = []

        for _ in range(self.n_estimators):
            X_sample, y_sample = bootstrap_sample(X, y)

            tree = MyDecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf
            )

            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        """
        Predict class labels using majority voting.
        """
        all_predictions = []

        for tree in self.trees:
            tree_pred = tree.predict(X)
            all_predictions.append(tree_pred)

        all_predictions = np.array(all_predictions)

        final_predictions = [
            majority_vote(all_predictions[:, i])
            for i in range(all_predictions.shape[1])
        ]

        return np.array(final_predictions)

    def score(self, X, y):
        """
        Compute classification accuracy.
        """
        y_pred = self.predict(X)
        return np.mean(y_pred == y)

In [7]:
# Random Forest Regressor

class RandomForestRegressor:
    """
    Random Forest model for regression.
    The model trains multiple decision trees on bootstrap samples
    and combines their predictions using averaging.
    """

    def __init__(self, n_estimators=10, max_depth=None, min_samples_leaf=5):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.trees = []

    def fit(self, X, y):
        """
        Train the random forest regressor.
        Each tree is trained on a different bootstrap sample.
        """
        self.trees = []

        for _ in range(self.n_estimators):
            X_sample, y_sample = bootstrap_sample(X, y)

            tree = MyDecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_leaf=self.min_samples_leaf
            )

            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        """
        Predict numeric values using the average prediction
        of all trees in the forest.
        """
        all_predictions = []

        for tree in self.trees:
            tree_pred = tree.predict(X)
            all_predictions.append(tree_pred)

        all_predictions = np.array(all_predictions)

        final_predictions = [
            average_predictions(all_predictions[:, i])
            for i in range(all_predictions.shape[1])
        ]

        return np.array(final_predictions)

**Evaluation metrics:**

Accuracy (for classification):
 **Accuracy** = (number of correct predictions) / (total number of samples)

MSE (Mean Squared Error, for regression):
 **MSE** = (1/n) * Σ (y_true - y_pred)^2

In [ ]:
# Train and evaluate Random Forest models

# ---------- Classification ----------
rf_clf = RandomForestClassifier(
    n_estimators=10,
    max_depth=5,
    min_samples_leaf=5
)

rf_clf.fit(X_train, y_train_class)

y_pred_class = rf_clf.predict(X_test)

# convert to numpy if needed
y_true_class = y_test_class.to_numpy() if hasattr(y_test_class, "to_numpy") else y_test_class

accuracy = np.mean(y_pred_class == y_true_class)

print("Classification Accuracy:", accuracy)


# ---------- Regression ----------
rf_reg = RandomForestRegressor(
    n_estimators=10,
    max_depth=5,
    min_samples_leaf=5
)

rf_reg.fit(X_train, y_train_reg)

y_pred_reg = rf_reg.predict(X_test)

# convert to numpy if needed
y_true_reg = y_test_reg.to_numpy() if hasattr(y_test_reg, "to_numpy") else y_test_reg

mse = np.mean((y_true_reg - y_pred_reg) ** 2)

print("Regression MSE:", mse)